In [2]:
# ============================================================
# Exercise 3: Cadence + Aktivitaetsklassifikation
# ============================================================

from pathlib import Path
import sys
import types

# DataProcessor importiert intern 'Excersice_3'.
# Der Shim verhindert einen Importfehler im Notebook-Kontext.
sys.modules.setdefault("Excersice_3", types.ModuleType("Excersice_3"))

from DataProcessor import DataProcessor
from scipy.signal import butter, filtfilt, welch
import numpy as np

# ------------------------------------------------------------
# 1) Hilfsfunktionen (Magnitude, Filter, Cadence, dominante Frequenz)
# ------------------------------------------------------------
def total_acc_magnitude(x, y, z):
    """Gesamtbeschleunigung (Magnitude) ohne DC-Anteil."""
    mag = np.sqrt(x**2 + y**2 + z**2)
    return mag - np.mean(mag)


def bandpass_filter(signal, fs, lowcut=0.5, highcut=5.0, order=4):
    """Bandpass im relevanten Schrittfrequenzbereich."""
    if len(signal) < 32:
        return signal
    nyquist = fs / 2.0
    low = max(lowcut / nyquist, 1e-6)
    high = min(highcut / nyquist, 0.999999)
    if not (0 < low < high < 1):
        return signal
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)


def estimate_cadence(block, fs):
    """Cadence in steps/min mit biased Autokorrelation."""
    if len(block) < 8:
        return 0.0
    block = block - np.mean(block)
    corr = np.correlate(block, block, mode="full")
    corr = corr[len(corr) // 2 :] / len(block)

    # Suchbereich: 0.8 Hz bis 3.5 Hz
    min_lag = max(1, int(fs / 3.5))
    max_lag = min(int(fs / 0.8), len(corr) - 1)
    if max_lag <= min_lag:
        return 0.0

    search = corr[min_lag : max_lag + 1]
    if search.size == 0:
        return 0.0

    peak_lag = int(np.argmax(search)) + min_lag
    return 60.0 * fs / peak_lag


def dominant_frequency(signal, fs):
    """Dominante Frequenz in Hz im Geh-/Laufbereich."""
    sig_filt = bandpass_filter(signal, fs)
    if len(sig_filt) < 16:
        return 0.0
    nperseg = int(min(2 * fs, len(sig_filt)))
    noverlap = int(min(nperseg // 2, nperseg - 1))
    freqs, pxx = welch(sig_filt, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mask = (freqs >= 0.5) & (freqs <= 3.5)
    if not np.any(mask):
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def classify_activity(cadence, dom_freq):
    """1=Ruhen, 2=Gehen, 3=Schnell Gehen, 4=Rennen."""
    if cadence < 70:
        return 1
    if cadence < 115:
        return 2
    if cadence < 150:
        # Hoehere dominante Frequenz im gleichen Cadence-Bereich deutet eher auf Rennen.
        return 4 if dom_freq >= 2.6 else 3
    return 4


def analyze_file(file_path, block_duration=5.0):
    """Analysiert eine pickle-Datei blockweise und gibt Cadence + Klassen zurueck."""
    dp = DataProcessor("rawdata/X22/")
    dp.loadRawData(str(file_path))

    devices = dp.getDevices()
    if not devices:
        return {"fs": 0, "cadences": [], "dom_freqs": [], "activities": []}

    # Erster verfuegbarer Sensorstream fuer konsistente Auswertung.
    dp.loadRawDataDevice(devices[0])
    x = dp.dfAcc["x"].values
    y = dp.dfAcc["y"].values
    z = dp.dfAcc["z"].values
    fs = int(dp.fs)

    block_samples = int(block_duration * fs)
    if block_samples <= 0:
        raise ValueError("block_duration muss > 0 sein")

    num_blocks = len(x) // block_samples
    if num_blocks == 0:
        return {"fs": fs, "cadences": [], "dom_freqs": [], "activities": []}

    cadences = []
    dom_freqs = []
    activities = []

    for b in range(num_blocks):
        start = b * block_samples
        end = start + block_samples
        mag = total_acc_magnitude(x[start:end], y[start:end], z[start:end])
        mag = bandpass_filter(mag, fs)

        cad = estimate_cadence(mag, fs)
        dom = dominant_frequency(mag, fs)
        act = classify_activity(cad, dom)

        cadences.append(float(cad))
        dom_freqs.append(float(dom))
        activities.append(int(act))

    return {
        "fs": fs,
        "cadences": cadences,
        "dom_freqs": dom_freqs,
        "activities": activities,
    }


ACTIVITY_NAMES = {1: "Ruhen", 2: "Gehen", 3: "Schnell Gehen", 4: "Rennen"}

# ------------------------------------------------------------
# 2) Aufgabe 3: Dateien aus rawdata/X22 laden und auswerten
# ------------------------------------------------------------
exercise3_files = [
    Path("rawdata/X22/normal_gehen3.pickle"),
    Path("rawdata/X22/schnell_Laufen10.pickle"),
    Path("rawdata/X22/rennen1.pickle"),
]

results_ex3 = {}
for file_path in exercise3_files:
    print(f"\n--- {file_path} ---")
    if not file_path.exists():
        print("Datei nicht gefunden, uebersprungen.")
        continue

    result = analyze_file(file_path, block_duration=5.0)
    results_ex3[str(file_path)] = result

    cadences = result["cadences"]
    activities = result["activities"]

    print(f"Samplingrate: {result['fs']} Hz")
    print(f"Anzahl 5s-Bloecke: {len(cadences)}")
    if cadences:
        print(f"Cadence (steps/min): {[f'{c:.1f}' for c in cadences]}")
        print(f"Aktivitaeten: {[(a, ACTIVITY_NAMES[a]) for a in activities]}")
        print(f"Mittlere Cadence: {np.mean(cadences):.1f} steps/min")
    else:
        print("Keine vollstaendigen 5s-Bloecke.")


--- rawdata\X22\normal_gehen3.pickle ---
Samplingrate: 200 Hz
Anzahl 5s-Bloecke: 7
Cadence (steps/min): ['67.4', '68.6', '70.2', '70.2', '70.6', '64.9', '210.5']
Aktivitaeten: [(1, 'Ruhen'), (1, 'Ruhen'), (2, 'Gehen'), (2, 'Gehen'), (2, 'Gehen'), (1, 'Ruhen'), (4, 'Rennen')]
Mittlere Cadence: 88.9 steps/min

--- rawdata\X22\schnell_Laufen10.pickle ---
Samplingrate: 200 Hz
Anzahl 5s-Bloecke: 13
Cadence (steps/min): ['73.6', '74.1', '73.6', '72.3', '69.4', '68.2', '69.0', '69.0', '68.2', '70.6', '72.3', '210.5', '210.5']
Aktivitaeten: [(2, 'Gehen'), (2, 'Gehen'), (2, 'Gehen'), (2, 'Gehen'), (1, 'Ruhen'), (1, 'Ruhen'), (1, 'Ruhen'), (1, 'Ruhen'), (1, 'Ruhen'), (2, 'Gehen'), (2, 'Gehen'), (4, 'Rennen'), (4, 'Rennen')]
Mittlere Cadence: 92.4 steps/min

--- rawdata\X22\rennen1.pickle ---
Samplingrate: 200 Hz
Anzahl 5s-Bloecke: 10
Cadence (steps/min): ['99.2', '179.1', '65.2', '83.3', '131.9', '210.5', '210.5', '48.0', '57.7', '210.5']
Aktivitaeten: [(2, 'Gehen'), (4, 'Rennen'), (1, 'Ruhen')

In [3]:
# ============================================================
# Exercise 4: Test des Algorithmus auf gelabelten Ordnerdaten
# ============================================================

from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


def infer_ground_truth_label(file_path):
    """Label aus Datei-/Ordnername ableiten."""
    path_lower = str(file_path).lower().replace("\\", "/")
    name_lower = file_path.name.lower()

    if "rennen" in path_lower:
        return 4
    if "schnell_gehen" in path_lower or "laufen_el" in path_lower or "schnell_laufen" in path_lower:
        return 3
    if "normal_gehen" in path_lower or name_lower.startswith("ex2_gehen"):
        return 2
    return None


dataset_root = Path("rawdata/X22/Excersice_2")
all_pickles = sorted(dataset_root.rglob("*.pickle"))
eval_files = [p for p in all_pickles if infer_ground_truth_label(p) is not None]

print(f"Gefundene Testdateien: {len(eval_files)}")

y_true = []
y_pred = []
per_file = []

for file_path in eval_files:
    gt_label = infer_ground_truth_label(file_path)
    result = analyze_file(file_path, block_duration=5.0)
    pred_blocks = result["activities"]
    num_blocks = len(pred_blocks)

    if num_blocks == 0:
        continue

    y_true.extend([gt_label] * num_blocks)
    y_pred.extend(pred_blocks)

    file_acc = float(np.mean(np.array(pred_blocks) == gt_label))
    per_file.append((file_path, gt_label, num_blocks, file_acc))

print(f"Ausgewertete Bloecke gesamt: {len(y_true)}")
print(f"Dateien mit mindestens 1 Block: {len(per_file)}")

if not y_true:
    print("Keine auswertbaren Bloecke gefunden.")
else:
    labels = sorted(set(y_true) | set(y_pred))
    label_names = [ACTIVITY_NAMES[l] for l in labels]

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=label_names,
        zero_division=0,
    )

    print(f"\nAccuracy (blockbasiert): {acc:.3f}")
    print("Labels:", labels, "->", label_names)
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(report)

    print("\nDateiuebersicht (erste 15):")
    for file_path, gt_label, n_blocks, file_acc in per_file[:15]:
        print(
            f"{file_path} | GT={gt_label} ({ACTIVITY_NAMES[gt_label]}) | "
            f"Bloecke={n_blocks} | Datei-Accuracy={file_acc:.3f}"
        )

ModuleNotFoundError: No module named 'sklearn'